In [67]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,ID,Symptom_Description,Age,Heart_Rate,Systolic_BP,O2_Saturation,Temperature,Target_Department,Urgency_Code
0,PAT_ABB4D396,Complaining of sharp palpitations in left arm....,22,167,117,93.2,36.8,Cardiology,2
1,PAT_FD013D59,Got bad tightness in my heart. Hurts since thi...,47,116,155,98.7,37.1,Cardiology,1
2,PAT_399A3F6C,Patient reports dull abdominal pain near gut. ...,90,89,111,96.0,36.6,Internal_Medicine,0
3,PAT_C4DFEA51,Experiencing bad shortness of breath in my che...,42,110,135,96.9,36.4,Respiratory,1
4,PAT_73056130,suffering from on and off tightness near heart...,21,62,118,98.4,36.2,Cardiology,0


In [68]:
df_train.columns

Index(['ID', 'Symptom_Description', 'Age', 'Heart_Rate', 'Systolic_BP',
       'O2_Saturation', 'Temperature', 'Target_Department', 'Urgency_Code'],
      dtype='object')

In [69]:
features = ['Age', 'Heart_Rate', 'Systolic_BP', 'O2_Saturation', 'Temperature']
target1 = 'Target_Department'
target2 = 'Urgency_Code'

X = df_train[features]
y1 = df_train[target1]
y2 = df_train[target2]

X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,Age,Heart_Rate,Systolic_BP,O2_Saturation,Temperature
0,-1.522945,3.822142,-0.361686,-1.303467,0.022526
1,-0.336139,1.451977,1.197172,0.490216,0.418693
2,1.705167,0.197185,-0.607821,-0.390319,-0.241585
3,-0.573500,1.173135,0.376720,-0.096807,-0.505696
4,-1.570417,-1.057608,-0.320663,0.392379,-0.769807


In [78]:
df_train.Symptom_Description[2]

"Patient reports dull abdominal pain near gut. It's suddenly."

In [79]:
import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return text

preprocess(df_train.Symptom_Description[2])

'patient reports dull abdominal pain near gut its suddenly'

In [80]:
df_train.Symptom_Description = df_train.Symptom_Description.apply(preprocess)
df_test.Symptom_Description = df_test.Symptom_Description.apply(preprocess)

df_train.head()

,ID,Symptom_Description,Age,Heart_Rate,Systolic_BP,O2_Saturation,Temperature,Target_Department,Urgency_Code
0,PAT_ABB4D396,complaining of sharp palpitations in left arm ...,22,167,117,93.2,36.8,Cardiology,2
1,PAT_FD013D59,got bad tightness in my heart hurts since this...,47,116,155,98.7,37.1,Cardiology,1
2,PAT_399A3F6C,patient reports dull abdominal pain near gut i...,90,89,111,96.0,36.6,Internal_Medicine,0
3,PAT_C4DFEA51,experiencing bad shortness of breath in my che...,42,110,135,96.9,36.4,Respiratory,1
4,PAT_73056130,suffering from on and off tightness near heart...,21,62,118,98.4,36.2,Cardiology,0


In [81]:
X_text = df_train['Symptom_Description']
X_text_final = df_test['Symptom_Description']

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer()
X_text = vect.fit_transform(X_text)
X_text_final = vect.transform(X_text_final)

X_text

<12000x217 sparse matrix of type '<class 'numpy.float64'>'
	with 117997 stored elements in Compressed Sparse Row format>

In [90]:
from scipy.sparse import hstack

X_combinat = hstack([X, X_text])
X_combinat_final = hstack([X_final, X_text_final])

In [96]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

model1 = LinearSVC()
model2 = XGBClassifier()

print(f"Subtask 1 score - {cross_val_score(model1, X_text, y1, cv=5, scoring='accuracy').mean()}")
print(f"Subtask 2 score - {cross_val_score(model2, X_combinat, y2, cv=5, scoring='f1_weighted').mean()}")

Subtask 1 score - 1.0
Subtask 2 score - 0.9988333312402886


In [97]:
model1.fit(X_text, y1)
preds_1 = model1.predict(X_text_final)

model2.fit(X_combinat, y2)
preds_2 = model2.predict(X_combinat_final)

In [98]:
df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.ID,
    "answer": preds_1
})

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.ID,
    "answer": preds_2
})

submission = pd.concat([df_1, df_2])

submission

,subtaskID,datapointID,answer
0,1,PAT_97CD10F5,Neurology
1,1,PAT_8C5956C1,Internal_Medicine
2,1,PAT_02434DD5,Orthopedics
3,1,PAT_3B38AD9E,Internal_Medicine
4,1,PAT_E52F936E,Orthopedics
...,...,...,...
2995,2,PAT_D676D218,0
2996,2,PAT_2668D3C8,0
2997,2,PAT_286BCA0B,2
2998,2,PAT_5878CC02,2


In [99]:
submission.to_csv("submission.csv", index=False)